# Barranquilla MOPAC: solids (Tv) on an A100

Validates the GPU support for materials on Colab (Runtime > Change runtime type > **A100 GPU**):

1. **check**: Si, quartz, rutile and urea cells of 144-192 atoms, 1SCF GRADIENTS with the device hcore, Fock and gradient compared to the CPU loops of the same run (`MOPAC_GPU_HCORE_CHECK`, `MOPAC_GPU_FOCK_CHECK`, `MOPAC_GPU_GRAD_CHECK`)
2. **medium**: quartz and rutile with 576 atoms and Si with 512 atoms, GPU time and heat compared to the Linux CPU references (4 vCPU: 240, 809, 1224 s)
3. **force**: diamond FORCE (64 atoms), GPU vs CPU (same binary, `MOPAC_NOGPU=1`)
4. **drc**: diamond DRC at 300 K, 40 steps, GPU vs CPU
5. stage profile of the medium cells (`MOPAC_GPU_PROFILE=1`)
6. optional: MOZYME (chignolin, crambin) against stored CPU references and the ctest suite

The source is cloned from GitHub (`juvenalyosa/mopac_gpu`, branch `main`). Everything is written to `/content/results` and downloaded as a zip in the last cell.

## 0. Options

In [ ]:
BRANCH = 'main'
RUN_CHECK = True        # 1: device hcore / Fock / gradient vs CPU (~5 min)
MEDIUM = 'quartz,silicon,rutile'   # 2: medium cells ('' to skip), ~2-6 min each
RUN_FORCE = True        # 3: diamond FORCE GPU and CPU (the CPU run takes ~10 min)
RUN_DRC = True          # 4: diamond DRC GPU and CPU (~4 min)
RUN_PROFILE = True      # 5: stage profile of the medium cells
RUN_MOZYME = False      # 6a: MOZYME chignolin / crambin vs stored CPU references (~15 min)
RUN_CTEST = False       # 6b: the MOPAC test suite with the GPU build (~20 min)

## 1. GPU

In [ ]:
import os, subprocess, time, re, json, shutil
from pathlib import Path

def run(cmd, log=None, check=True, env=None):
    """Run a shell command, stream its output, keep it in log."""
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'a') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

RES = Path('/content/results'); RES.mkdir(parents=True, exist_ok=True)
run('nvidia-smi', RES / 'nvidia-smi.txt')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu, '| CPUs:', os.cpu_count())
if 'A100' not in gpu:
    print('WARNING: not an A100 (Runtime > Change runtime type). The checks still run.')
CC = gpu.split(',')[-1].strip().replace('.', '')

## 2. Build (CUDA, Release)

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q --depth 1 -b {BRANCH} https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
commit = subprocess.run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short', shell=True,
                        capture_output=True, text=True).stdout.strip()
print('commit:', commit)
(RES / 'commit.txt').write_text(commit + '\n' + gpu + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'> {RES}/cmake.log 2>&1 || (tail -40 {RES}/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {os.cpu_count()} > {RES}/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error {RES}/build.log | head -80; exit 1)')
MOPAC = BUILD / 'mopac'
assert MOPAC.exists()
print(f'build OK in {time.time() - t0:.0f} s: {MOPAC}')
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count())
CHECK = f'python3 {SRC}/tests/check_materials_gpu.py {MOPAC} --work-dir {RES}/materials'

## 3. Device hcore / Fock / gradient vs CPU (144-192 atoms)

In [ ]:
if RUN_CHECK:
    run(f"{CHECK} --medium '' --skip-force --skip-drc", RES / 'check.log', check=False)

## 4. Medium cells (512-576 atoms) vs CPU references

In [ ]:
if MEDIUM:
    run(f"{CHECK} --skip-check --medium {MEDIUM} --skip-force --skip-drc", RES / 'medium.log', check=False)

## 5. Diamond FORCE, GPU vs CPU

In [ ]:
if RUN_FORCE:
    run(f"{CHECK} --skip-check --medium '' --skip-drc", RES / 'force.log', check=False)

## 6. Diamond DRC (300 K), GPU vs CPU

In [ ]:
if RUN_DRC:
    run(f"{CHECK} --skip-check --medium '' --skip-force", RES / 'drc.log', check=False)

## 7. Stage profile of the medium cells

In [ ]:
def profile(path):
    out = Path(path).read_text(errors='ignore').splitlines()
    keep = [l.strip() for l in out if '[PROFILE]' in l and any(k in l for k in ('conv_', 'hcore_total', 'deriv', 'dcart'))]
    keep += [l.strip() for l in out if 'AFTER  HCORE' in l or 'AFTER  DERIV' in l or 'AFTER  ITER' in l]
    keep += [l.strip() for l in out if 'AFTER  FOCKS' in l][-1:] + [l.strip() for l in out if 'AFTER  FULL DIAG' in l][-1:]
    keep += [l.strip() for l in out if 'FINAL HEAT' in l or 'TOTAL JOB TIME' in l]
    keep.append(f"iterations {sum('TIME FOR ITERATION' in l for l in out)}")
    return keep

if RUN_PROFILE and MEDIUM:
    with open(RES / 'profile.txt', 'w') as fh:
        for name in MEDIUM.split(','):
            f = RES / 'materials' / 'medium' / name / f'{name}.out'
            if f.exists():
                lines = [f'== {name}'] + ['   ' + l for l in profile(f)]
                print('\n'.join(lines)); fh.write('\n'.join(lines) + '\n')

## 8. Optional: MOZYME references and ctest

In [ ]:
if RUN_MOZYME:
    run(f'python3 {SRC}/tests/check_mozyme_gpu_reference.py {MOPAC} --data {SRC}/tests/data/mozyme_gpu_reference '
        f'--work-dir {RES}/mozyme --workers 0 --concurrent 0 --skip-forcets', RES / 'mozyme.log', check=False)
if RUN_CTEST:
    run('pip -q install numpy')
    run(f'cmake -S {SRC} -B {BUILD} -DTESTS=ON > {RES}/ctest_configure.log 2>&1 && '
        f'cmake --build {BUILD} --parallel {os.cpu_count()} > {RES}/ctest_build.log 2>&1', check=False)
    run(f'cd {BUILD} && ctest -j {os.cpu_count()} --output-on-failure 2>&1 | tail -40', RES / 'ctest.log', check=False)

## 9. Summary and download

In [ ]:
lines = [f'commit: {commit}', f'gpu: {gpu}']
for log in ('check', 'medium', 'force', 'drc', 'mozyme', 'ctest'):
    p = RES / f'{log}.log'
    if p.exists():
        lines += [l.rstrip() for l in p.read_text(errors='ignore').splitlines()
                  if l.startswith(('PASS', 'FAIL', '[materials]', 'all checks', '[ctest]')) or 'tests passed' in l
                  or 'check(s) failed' in l]
summary = '\n'.join(lines)
print(summary)
(RES / 'SUMMARY.txt').write_text(summary + '\n')
shutil.make_archive('/content/materials_a100_results', 'zip', RES)
from google.colab import files
files.download('/content/materials_a100_results.zip')